In [10]:
import pandas as pd
import yaml

csv_name = input("Введите название csv файла (например: 2026-03-19_17-13-07): ")

df = pd.read_csv(csv_name+'.csv', parse_dates=['Время, ГГГГ-ММ-ДД ЧЧ:ММ:СС'])
df.columns = ['Время', 'Температура', 'Влажность', 'Осадки', 'Скорость ветра', 'city_id']
df['Дата'] = df['Время'].dt.date

# для merge
ref_cities = pd.read_csv("cities.csv")

# Средняя температура по дням
T_mean = df.groupby('Дата')['Температура'].mean().round(1)

# Суммарные осадки по дням
P_sum = df.groupby('Дата')['Осадки'].sum()

# Количество часов с осадками > 0 по дням
daily_rainy_hours = df[df['Осадки'] > 0].groupby('Дата').size()

# Создание основного DataFrame с KPI
with open("variant_4.yml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)
    kpi_df = pd.DataFrame({
        'Средняя температура': T_mean,
        'Сумма осадков': P_sum,
        'Кол-во часов с осадками': daily_rainy_hours,
        'city_id' : config["entity"]["city_id"]
})

# Заполнение NaN значений для дней без осадков
kpi_df['Кол-во часов с осадками'] = kpi_df['Кол-во часов с осадками'].fillna(0).astype(int)
kpi_df = kpi_df.reset_index()

#merge для daily-kpi
merged_kpi_df = kpi_df.merge(
    ref_cities,
    on = "city_id",
    how = "left"
)
display(merged_kpi_df)

merged_kpi_df.to_csv('mart_daily_'+csv_name+'.csv', index=False, encoding='utf-8')
print("KPI по дням сохранены в файл: ", 'mart_daily_', csv_name, '.csv', sep='')

# Топ-5 дней по максимальной скорости ветра
daily_max_wind = df.groupby('Дата')['Скорость ветра'].max().reset_index()
# Сортируем по убыванию и берем топ-5
top5_windy_days = daily_max_wind.nlargest(5, 'Скорость ветра').reset_index(drop=True)
top5_windy_days.columns = ['Дата', 'Cкорость ветра']

#merge для топ-5
with open("variant_4.yml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)
    top5_windy_days['city_id'] = config['entity']['city_id']
merged_top5_windy_days = top5_windy_days.merge(
    ref_cities,
    on = "city_id",
    how = "left"
)

print("\nТоп-5 дней по максимальной скорости ветра:")
display(merged_top5_windy_days)

merged_top5_windy_days.to_csv('mart_top_'+csv_name+'.csv', index=False, encoding='utf-8')
print("\nТоп-5 дней по максимальной скорости ветра сохранены в файл: ", "mart_top_", csv_name, ".csv", sep="")

Введите название csv файла (например: 2026-03-19_17-13-07):  2026-03-19_17-13-07


,Дата,Средняя температура,Сумма осадков,Кол-во часов с осадками,city_id,city_name,country_code,latitude,longitude,timezone
0,2026-03-01,9.7,0.6,5,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
1,2026-03-02,11.1,0.0,0,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
2,2026-03-03,9.2,0.0,0,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
3,2026-03-04,8.5,0.0,0,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
4,2026-03-05,11.6,0.0,0,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
5,2026-03-06,9.1,5.4,7,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
6,2026-03-07,8.1,0.0,0,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
7,2026-03-08,8.8,0.0,0,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
8,2026-03-09,9.7,0.2,1,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
9,2026-03-10,10.1,0.4,3,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London


KPI по дням сохранены в файл: mart_daily_2026-03-19_17-13-07.csv

Топ-5 дней по максимальной скорости ветра:


,Дата,Cкорость ветра,city_id,city_name,country_code,latitude,longitude,timezone
0,2026-03-12,33.1,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
1,2026-03-13,28.1,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
2,2026-03-15,26.1,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
3,2026-03-01,21.9,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London
4,2026-03-11,21.7,GB_LON,Лондон,GB,51.5072,-0.1276,Europe/London



Топ-5 дней по максимальной скорости ветра сохранены в файл: mart_top_2026-03-19_17-13-07.csv
